# Progressive Tables (`window.craftTables`)

> Enhances Markdown, pandas, polars, and DuckDB tables with a keyboard-scrollable wrapper, automatic right-aligned numeric columns, click-to-sort flat headers, and a subtle **Copy TSV** button.

## Table Templates (`#| export_ft`)

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = dict(
    table_wrap=Div(klass='craft-table-wrap', tabindex='0', role='region', aria_label='Scrollable table'),
    table_copy_btn=Button('Copy TSV', type='button', klass='craft-table-copy', aria_label='Copy table as TSV'),
)

## Table Enhancer (`window.craftTables`)

Watches `#dialog-container` via `window.__craftCMCore.onDomChange` and progressively enhances each unenhanced table in place.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Progressive table enhancer (wrap, numeric align, flat-table sort, copy TSV) ── */
(() => {
  window.craftTables?.disable?.();
  const NUM_RE = /^[-+\u2212]?[$€£¥]?\s*\d[\d,_]*(?:\.\d+)?(?:[eE][+-]?\d+)?%?$/;
  const EMPTY_RE = /^(?:|nan|na|n\/a|null|none|-|\u2014)$/i;
  const parseNum = s => {
    const t = s.trim();
    if (!t || EMPTY_RE.test(t)) return null;
    if (!NUM_RE.test(t)) return NaN;
    return parseFloat(t.replace(/[\u2212]/g, '-').replace(/[$€£¥,\s_%]/g, ''));
  };

  const enhanceTable = tbl => {
    if (!tbl || tbl.dataset.craftTable) return;
    tbl.dataset.craftTable = '1';
    tbl.classList.add('uk-table', 'uk-table-small', 'uk-table-divider', 'uk-table-hover');

    // 1. Wrap in a keyboard-scrollable container if not already wrapped
    let wrap = tbl.parentElement;
    if (!wrap?.classList.contains('craft-table-wrap')) {
      wrap = window.__craftCMCore.mountTpl('table_wrap');
      tbl.parentNode.insertBefore(wrap, tbl);
      wrap.appendChild(tbl);
    }

    const headRows = [...tbl.querySelectorAll(':scope > thead > tr')];
    const tbody = tbl.querySelector(':scope > tbody');
    const bodyRows = tbody ? [...tbody.querySelectorAll(':scope > tr')] : [];
    const hasSpans = [...tbl.querySelectorAll('th, td')].some(c => c.colSpan > 1 || c.rowSpan > 1);

    // 2. Detect numeric columns and right-align both header and cells
    if (!hasSpans && bodyRows.length) {
      const sample = bodyRows.slice(0, 40);
      const colCount = sample[0].children.length;
      const headCells = headRows.length === 1 ? [...headRows[0].children] : null;
      for (let col = 0; col < colCount; col++) {
        let numHits = 0, nonEmpty = 0;
        for (const row of sample) {
          const cell = row.children[col];
          if (!cell || cell.tagName === 'TH') continue;
          const v = parseNum(cell.textContent || '');
          if (v === null) continue;
          nonEmpty++;
          if (!Number.isNaN(v)) numHits++;
        }
        if (nonEmpty > 0 && numHits === nonEmpty) {
          for (const row of bodyRows) row.children[col]?.classList.add('craft-num');
          if (headCells && headCells[col]) headCells[col].classList.add('craft-num');
        }
      }
    } else if (hasSpans && bodyRows.length) {
      // MultiIndex: align numeric <td> cells (and matching top-row headers) without disturbing rowspan
      for (const row of bodyRows) {
        for (const cell of row.children) {
          if (cell.tagName !== 'TD') continue;
          const v = parseNum(cell.textContent || '');
          if (v !== null && !Number.isNaN(v)) cell.classList.add('craft-num');
        }
      }
      if (headRows.length && headRows[0].children.length === bodyRows[0].children.length) {
        for (let col = 0; col < bodyRows[0].children.length; col++) {
          if (bodyRows[0].children[col]?.classList.contains('craft-num')) {
            headRows[0].children[col]?.classList.add('craft-num');
          }
        }
      }
    }

    // 3. Click/keyboard sort on flat tables (>= 2 rows, single header row, no spans)
    if (!hasSpans && headRows.length === 1 && bodyRows.length >= 2) {
      const origOrder = bodyRows.slice();
      const ths = [...headRows[0].children];
      ths.forEach((th, colIdx) => {
        if (!(th.textContent || '').trim()) return;
        th.classList.add('craft-sort');
        th.tabIndex = 0;
        th.setAttribute('aria-sort', 'none');
        const toggle = () => {
          const cur = th.getAttribute('aria-sort');
          const next = cur === 'ascending' ? 'descending' : cur === 'descending' ? 'none' : 'ascending';
          for (const other of ths) if (other !== th && other.classList.contains('craft-sort')) other.setAttribute('aria-sort', 'none');
          th.setAttribute('aria-sort', next);
          if (next === 'none') {
            tbody.append(...origOrder);
            return;
          }
          const dir = next === 'ascending' ? 1 : -1;
          const isNum = th.classList.contains('craft-num');
          const sorted = origOrder.slice().sort((a, b) => {
            const ta = (a.children[colIdx]?.textContent || '').trim();
            const tb = (b.children[colIdx]?.textContent || '').trim();
            if (isNum) {
              const na = parseNum(ta), nb = parseNum(tb);
              const va = (na === null || Number.isNaN(na)) ? Infinity : na;
              const vb = (nb === null || Number.isNaN(nb)) ? Infinity : nb;
              return (va - vb) * dir;
            }
            return ta.localeCompare(tb, undefined, { numeric: true, sensitivity: 'base' }) * dir;
          });
          tbody.append(...sorted);
        };
        th.addEventListener('click', toggle);
        th.addEventListener('keydown', e => {
          if (e.key === 'Enter' || e.key === ' ') { e.preventDefault(); toggle(); }
        });
      });
    }

    // 4. Subtle Copy TSV button on output tables
    if (tbl.closest('.card-out') && !wrap.querySelector(':scope > .craft-table-copy')) {
      const btn = window.__craftCMCore.mountTpl('table_copy_btn');
      btn.addEventListener('click', async e => {
        e.stopPropagation();
        const rows = [...tbl.querySelectorAll('tr')].map(tr =>
          [...tr.children].map(c => (c.textContent || '').replace(/[\t\r\n]+/g, ' ').trim()).join('\t')
        ).join('\n');
        try {
          await navigator.clipboard.writeText(rows);
          btn.textContent = 'Copied';
          btn.classList.add('is-copied');
          setTimeout(() => { btn.textContent = 'Copy TSV'; btn.classList.remove('is-copied'); }, 1400);
        } catch {}
      });
      wrap.insertBefore(btn, tbl);
    }
  };

  const enhanceAll = () => {
    for (const tbl of document.querySelectorAll('#dialog-container .msg-content table:not([data-craft-table])')) {
      enhanceTable(tbl);
    }
  };
  let offDom = null;
  const start = () => {
    offDom = window.__craftCMCore.onDomChange(enhanceAll, {delay: 80, inDialog: true});
    enhanceAll();
  };
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', start);
  else start();
  window.craftTables = {
    enhance: enhanceAll,
    disable() {
      offDom?.();
      offDom = null;
      document.removeEventListener('DOMContentLoaded', start);
    },
  };
})();

## Table CSS (Dark & Light Modes)

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── 16. Cohesive Tabular Output (FrankenUI .uk-table* + CRAFT sticky/sort/TSV) ── */
.editable > .msg-card,
.editable > .msg-card > .msg-body,
.editable > .msg-card .msg-content,
.editable > .msg-card .msg-content .prose { min-width: 0; max-width: 100%; }
.craft-table-wrap {
  position: relative; width: fit-content; max-width: 100%; max-height: 480px;
  overflow: auto; -webkit-overflow-scrolling: touch; margin: 0.4em 0 !important;
  border: 1px solid var(--craft-border); border-radius: 5px; background: var(--craft-surface);
}
.craft-table-wrap:focus-visible { outline: 1.5px solid var(--craft-accent); outline-offset: 1px; }
.msg-content table {
  --uk-table-width: auto;
  --uk-table-collapse: separate;
  --uk-table-cell-padding: 4px 10px;
  --uk-table-header-font-size: 11px;
  --uk-table-divider-border: 1px solid var(--craft-border-soft);
  --uk-table-hover-bg: var(--craft-row-hover);
  margin: 0 !important; border: 0 !important; background: transparent !important;
  font-family: var(--craft-prose-family) !important; font-size: 11px !important; line-height: 1.4 !important;
  font-stretch: normal !important; font-feature-settings: "tnum" 1, "lnum" 1, "onum" 0 !important;
  font-variant-numeric: tabular-nums lining-nums !important; color: var(--craft-fg) !important;
}
.msg-content table thead { position: sticky; top: 0; z-index: 2; }
.msg-content table thead th {
  background: var(--craft-recessed) !important; color: var(--craft-fg-strong) !important;
  font-weight: 600 !important; padding: 5px 10px !important;
  border-bottom: 1px solid var(--craft-border) !important; vertical-align: bottom !important; white-space: nowrap;
}
.msg-content table tbody th {
  background: var(--craft-surface-alt) !important; color: var(--craft-fg) !important;
  font-weight: 500 !important; border-right: 1px solid var(--craft-border) !important; white-space: nowrap;
}
.msg-content table :is(th, td).craft-num { text-align: right !important; white-space: nowrap; }
.msg-content table thead th.craft-sort { cursor: pointer; user-select: none; padding-right: 18px !important; position: relative; }
.msg-content table thead th.craft-sort:hover { background: var(--craft-hover) !important; }
.msg-content table thead th.craft-sort:focus-visible { outline: 1.5px solid var(--craft-accent); outline-offset: -1.5px; }
.msg-content table thead th.craft-sort::after { content: ""; position: absolute; right: 5px; top: 50%; transform: translateY(-50%); font-size: 8px; color: var(--craft-accent); opacity: 0; }
.msg-content table thead th.craft-sort[aria-sort="ascending"]::after { content: "\25B2"; opacity: 1; }
.msg-content table thead th.craft-sort[aria-sort="descending"]::after { content: "\25BC"; opacity: 1; }
.craft-table-copy {
  position: sticky; float: right; top: 3px; right: 4px; z-index: 4; margin-left: -60px;
  padding: 1px 6px; border: 1px solid var(--craft-border); border-radius: 3px;
  background: var(--craft-copy-bg); color: var(--craft-fg);
  font-family: var(--craft-prose-family); font-size: 9.5px; line-height: 16px;
  cursor: pointer; opacity: 0; transition: opacity 0.12s ease, color 0.12s ease, border-color 0.12s ease;
}
.craft-table-wrap:is(:hover, :focus-within) .craft-table-copy, .craft-table-copy.is-copied { opacity: 1; }
.craft-table-copy:is(:hover, :focus-visible) { opacity: 1; color: var(--craft-fg-strong); border-color: var(--craft-accent); outline: 2px solid var(--craft-accent); outline-offset: 1px; }
.craft-table-copy.is-copied { color: var(--craft-green); border-color: var(--craft-green); }

Verify `08_tables.ipynb` exports `window.craftTables` and table CSS.

In [ ]:
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/08_tables.ipynb')
assert 'window.craftTables' in '\n'.join(code for _, code in assets['CRAFT.js'])
assert '.craft-table-wrap' in '\n'.join(code for _, code in assets['CRAFT.css'])